### Libraries

In [2]:
pip install numpy matplotlib

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import time
import pickle
from pathlib import Path

### Directories

In [4]:
data_dir = Path("..") / "data"
polygon_dir = Path("..") / "polygons"

In [5]:
# create output directories if they do not already exist
figure_dir = Path("..") / "figures"
for dir in [figure_dir]:
    if not dir.is_dir():
        dir.mkdir(parents=True, exist_ok=True)

if not polygon_dir.is_dir():
    from polygonanalyser import xs_all, ys_all, xs_yb, ys_yb


### Video load and show

In [6]:
# load the video
video_filename = "Video_Test_F2787_125743_01_VIDCKPT_sec.mpg"
video_path = data_dir / video_filename 
cap = cv2.VideoCapture(video_path)

In [7]:
show_video = 0

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
window_name = "flight"

cv2.namedWindow(window_name)

# method for setting frame
def on_trackbar_change(trackbar_value):
    cap.set(cv2.CAP_PROP_POS_FRAMES, trackbar_value)
    return 

cv2.createTrackbar("...", window_name, 0, total_frames - 1, on_trackbar_change)

while cap.isOpened():
    if not show_video: break

    current_trackbar_pos = cv2.getTrackbarPos("...", window_name)

    ret, frame = cap.read()
 
    # if frame is read correctly ret is True. ret will be false when video is over
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break

    # update the trackbar regularly
    current_frame_id = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
    cv2.setTrackbarPos("...", window_name, current_frame_id)

    # gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cv2.imshow('frame', frame)#, gray)

    # actions to do
    # quit
    if cv2.waitKey(1) == ord('q'):
        break

cv2.destroyAllWindows()

### Mask creation

In [8]:
# set first frame and get the shape
cap.set(cv2.CAP_PROP_POS_FRAMES, 0) 
_, first_frame = cap.read()
frame_shape = first_frame.shape

# define method for determining whether a point is inside a polygon
# https://www.geeksforgeeks.org/dsa/how-to-check-if-a-given-point-lies-inside-a-polygon/
def on_segment(x1, y1, x2, y2, x, y):
    c = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)

    if c != 0:
        return False

    return (min(x1, x2) <= x <= max(x1, x2) and
            min(y1, y2) <= y <= max(y1, y2))

def isInside(arr, x, y):
    n = len(arr)
    inside = False

    j = n - 1

    for i in range(n):
        x1, y1 = arr[i]
        x2, y2 = arr[j]

        # Point lies on the current edge
        if on_segment(x1, y1, x2, y2, x, y):
            return True

        # Check whether the horizontal ray from (x, y)
        # intersects the current edge
        intersect = ((y1 > y) != (y2 > y) and
                     x < (x2 - x1) * (y - y1) / (y2 - y1) + x1)

        if intersect:
            inside = not inside

        j = i

    return inside

# fit the polygons to match position of apparatus in frame
def fit_polygon_all_apparatus(xs, ys):
    # shift polygons to upper right corner
    xs -= np.min(xs)
    ys -= np.min(ys)

    # move to proper startting position
    x_move = frame_shape[1]//2 - 10
    y_move = frame_shape[0]//2 + 85
    xs += x_move 
    ys += y_move 
    return xs, ys

def fit_polygon_yellow_black_apparatus(xs, ys):
    # shift polygons to upper right corner
    xs -= np.min(xs)
    ys -= np.min(ys)

    # move to proper startting position
    x_move = frame_shape[1]//2 + 100
    y_move = frame_shape[0]//2 + 85
    xs += x_move 
    ys += y_move 
    return xs, ys

# load polygons and fit polygons
xs_all = np.load(polygon_dir / "xs_all.npy")
ys_all = np.load(polygon_dir / "ys_all.npy")
xs_yb = np.load(polygon_dir / "xs_yb.npy")
ys_yb = np.load(polygon_dir / "ys_yb.npy")
xs_all, ys_all = fit_polygon_all_apparatus(xs_all, ys_all)
xs_yb, ys_yb = fit_polygon_yellow_black_apparatus(xs_yb, ys_yb)

# convert polygons to list of points
polygon = list(zip(xs_all, ys_all))
polygon = list(zip(xs_yb, ys_yb))

# draw the polygon on the first frame
previous_point = polygon[0]
for point in polygon[1:]:
    cv2.line(first_frame, previous_point, point, (255, 0, 0), 5)
    previous_point = point
cv2.line(first_frame, polygon[-1], polygon[0], (255, 0, 0), 5)

# create a mask using the polygon
mask = np.zeros(frame_shape[:2], dtype=np.uint8)
cv2.fillPoly(
    mask, 
    [np.asarray(polygon, dtype=np.int32)], 
    1
)

# plot the masked image along with the drawing of the polygon
first_frame_masked = first_frame * mask[:, :, None]
cv2.imwrite(Path("..") /"figures" / "image.png", first_frame_masked)




True

### Singular frames 

In [9]:
# define bins used for all histograms
bins = np.arange(0, 256, 5)

def normalize_brightness(frame):
    # normalization using LAB frame
    # https://en.wikipedia.org/wiki/CIELAB_color_space
    # L in LAB is "lightness"
    lab_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab_frame)

    # extract the valid l-channel pixels
    valid_l_pixels = l_channel[mask==1].astype(np.float32)

    # normalize brightness channel by mean=255//2, std=50?
    l_mean, l_std = valid_l_pixels.mean(), valid_l_pixels.std()

    normalized_l = (valid_l_pixels - l_mean) / l_std * 50 + 128
    normalized_l = np.clip(normalized_l, 0, 255).astype(np.uint8)
    l_channel[mask==1] = normalized_l

    # merge back
    normalized_LAB = cv2.merge([l_channel, a_channel, b_channel])
    normalized = cv2.cvtColor(normalized_LAB, cv2.COLOR_LAB2BGR)
    return normalized

def extract_im_and_hist(video_capture, target_frame_index, mask, savefigs=True, searchup=True):
    """
    load a particular frame index in the video capture
    """
    if searchup:
        video_capture.set(cv2.CAP_PROP_POS_FRAMES, target_frame_index) # search up the frame
    ret, frame = cap.read()

    if not ret:
        return 

    # filter the frame and normalize it
    frame_filtered = frame * mask[:, :, None]

    # normalize
    frame_filtered = normalize_brightness(frame_filtered)

    # pick out the valid pixels
    valid_pixels = frame_filtered[mask == 1]

    # get counts (density)
    counts_blue, _ = np.histogram(valid_pixels[:, 0], bins, density=True)
    counts_green, _ = np.histogram(valid_pixels[:, 1], bins, density=True)
    counts_red, _ = np.histogram(valid_pixels[:, 2], bins, density=True)

    # window_length = 15
    # polyorder = 3
    # height_threshold = 0.005
    # filtered_blue = savgol_filter(counts_blue, window_length, polyorder)
    # filtered_green = savgol_filter(counts_green, window_length, polyorder)
    # filtered_red = savgol_filter(counts_red, window_length, polyorder)
    # peaks_blue = find_peaks(filtered_blue, height_threshold)
    # peaks_green = find_peaks(filtered_green, height_threshold)
    # peaks_red = find_peaks(filtered_red, height_threshold)
    

    if savefigs:
        fig, axs=plt.subplots(3, 1)
        # axs[0].plot(bins[:-1], filtered_blue)
        # axs[1].plot(bins[:-1], filtered_green)
        # axs[2].plot(bins[:-1], filtered_red)

        # axs[0].scatter(peaks_blue[0], peaks_blue[1]["peak_heights"], c="r")
        # axs[1].scatter(peaks_green[0], peaks_green[1]["peak_heights"], c="r")
        # axs[2].scatter(peaks_red[0], peaks_red[1]["peak_heights"], c="r")
        axs[0].set_title("blue", c="blue")
        axs[1].set_title("green", c="green")
        axs[2].set_title("red", c="red")
        fig.tight_layout()

        fig.savefig(Path("..") / "figures" / "histogram.png")
        plt.close()

        # backconvert frame to integers
        
        cv2.imwrite(Path("..") / "figures" / "image.png", frame_filtered)

    return {
        "counts_blue": counts_blue,
        "counts_green": counts_green,
        "counts_red": counts_red,
    }

# target_frame = 10000
# target_frame = 4000
target_frame = 1000
extract_im_and_hist(cap, target_frame, mask, savefigs=1)

{'counts_blue': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        8.14190172e-05, 2.40379955e-03, 8.63429291e-03, 1.70979936e-02,
        1.44925851e-02, 1.36396239e-02, 1.12280702e-02, 9.79742173e-03,
        9.47174566e-03, 1.64195018e-02, 1.59581274e-02, 1.99941844e-02,
        1.10458467e-02, 1.03363381e-02, 9.97189105e-03, 8.12639333e-03,
        5.62954347e-03, 8.41717554e-03, 3.67936416e-03, 1.37249200e-03,
        8.99486285e-04, 4.69128623e-04, 3.21798973e-04, 1.93854803e-04,
        1.04681593e-04, 4.65251527e-05, 2.71396724e-05, 3.87709606e-05,
        4.26480566e-05, 5.81564408e-05, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]),
 'counts_green': array([0.00000000e+00, 0.00000000e+00, 0.00000

### Define ice / noice regions 

In [10]:
target_frame = 10000 - 1000
for addition in range(0, 10000, 10):
    break    
    extract_im_and_hist(cap, target_frame+addition, mask)
    time.sleep(0.05)
target_frame = 0

for addition in range(0, 5000, 100):
    break
    extract_im_and_hist(cap, target_frame+addition, mask=np.ones(frame_shape))
    time.sleep(0.1)

icing_frames = 10000-1000, 10000-1000 + 10000
noicing_frames = 0, 5000

In [11]:
extract_counts = 0

def extract_counts_icing(counts):
    # extract counts from the icing frames
    for i in np.arange(icing_frames[0], icing_frames[1], 100):
        histogram_counts = extract_im_and_hist(cap, i, mask, savefigs=True)
        for key in histogram_counts:
            counts["icing"][key].append(histogram_counts[key])
            # time.sleep(0.1)

def extract_counts_no_icing(counts):
    # extract counts from the no-icing frames
    for i in np.arange(noicing_frames[0], noicing_frames[1], 100):
        histogram_counts = extract_im_and_hist(cap, i, mask, savefigs=True)
        for key in histogram_counts:
            counts["noicing"][key].append(histogram_counts[key])
            # time.sleep(0.1)
    

if extract_counts:
    counts = {
        "icing": {
            "counts_blue": [],
            "counts_green": [],
            "counts_red": []
        },
        "noicing": {
            "counts_blue": [],
            "counts_green": [],
            "counts_red": []
        },
    }

    extract_counts_icing(counts)
    extract_counts_icing(counts)
    # convert the lists to numpy arrays
    for key in counts:
        for color in counts[key]:
            counts[key][color] = np.array(counts[key][color])

In [12]:
get_average_counts = 0
if get_average_counts:
    average_dist_dir = Path("..") / "average_dist"
    if not average_dist_dir.is_dir(): average_dist_dir.mkdir(parents=True, exist_ok=True)

    # find average histograms and plot them
    figicing, axsicing = plt.subplots(3, 1)
    for i, key in enumerate(counts["icing"].keys()):
        average_count = counts["icing"][key].mean(axis=0)
        np.save(average_dist_dir / ("icing" + key + ".npy"), average_count)
        axsicing[i].plot(bins[:-1], average_count)

    fignoicing, axsnoicing = plt.subplots(3, 1)
    for i, key in enumerate(counts["noicing"].keys()):
        average_count = counts["noicing"][key].mean(axis=0)
        np.save(average_dist_dir / ("noicing" + key + ".npy"), average_count)
        axsnoicing[i].plot(bins[:-1], average_count)


    # formatting
    for i, color in enumerate(["blue", "green", "red"]):
        axsicing[i].set_title(color, c=color)
        axsnoicing[i].set_title(color, c=color)

    figicing.suptitle("Icing")
    fignoicing.suptitle("No icing")
    figicing.tight_layout()
    fignoicing.tight_layout()
    figicing.savefig(figure_dir / "Mean icing histogram")
    fignoicing.savefig(figure_dir / "Mean no-icing histogram")

    plt.close("all")


In [13]:
do_savgol_peak_analysis = 0
if do_savgol_peak_analysis:
    counts = {
        "icingblue": np.load(average_dist_dir / "icingcounts_blue.npy"),
        "icinggreen": np.load(average_dist_dir / "icingcounts_green.npy"),
        "icingred": np.load(average_dist_dir / "icingcounts_red.npy"),
        "noicingblue": np.load(average_dist_dir / "noicingcounts_blue.npy"),
        "noicinggreen": np.load(average_dist_dir / "noicingcounts_green.npy"),
        "noicingred": np.load(average_dist_dir / "noicingcounts_red.npy"),
    }


    fig, axs=plt.subplots(3, 2)
    # fig2, axs2 = plt.subplots(3, 2)
    std0 = 5
    p04 = [50, 150, std0, std0]
    p02 = [128, std0]

    boundsmu0 = (40, 210)
    boundssigma0 = (5, 10)
    bounds4 = [boundsmu0]*2 + [boundssigma0]*2
    bounds2 = [boundsmu0] + [boundssigma0]

    def curve4(x, mu1, mu2, sigma1, sigma2):
        out = ss.norm(mu1, sigma1).pdf(x) + ss.norm(mu2, sigma2).pdf(x)
        outpositive = np.where(0 < x, out, 0.0)
        return np.where(x < 254, outpositive, 0.0)

    def curve2(x, mu, sigma):
        out = ss.norm(mu, sigma).pdf(x)
        return np.where(0 < x, out, 0.0)

    # def curve(x, a1, a2):
    #     return (ss.lognorm(a1).pdf(x) + 
    #             ss.lognorm(a2).pdf(x))

    # for i, key in enumerate(counts.keys()):
    #     ax = axs.T.flatten()[i]
    #     popt4, _ = curve_fit(curve4, bins[:-1], counts[key], p04)
    #     popt2, _ = curve_fit(curve2, bins[:-1], counts[key], p02)
    #     ax.plot(bins[:-1], counts[key])
    #     ax.plot(bins[:-1], curve4(bins[:-1], *popt4))
    #     ax.plot(bins[:-1], curve2(bins[:-1], *popt2))
    #     ax.set_title(key)
    #     ax.set_xticks(range(1, 255, 35))
    for i, key in enumerate(counts.keys()):
        ax = axs.T.flatten()[i]
        # popt4, _ = curve_fit(curve4, bins[:-1], counts[key], p04)
        # popt2, _ = curve_fit(curve2, bins[:-1], counts[key], p02)
        # objective4 = lambda p: np.sum((counts[key] - curve4(bins[:-1], *p))**2)
        # result = differential_evolution(objective4, bounds4)
        # popt4 = result.x
        filtered = savgol_filter(counts[key], 35, 5)
        filtered = counts[key]
        ax.plot(bins[:-1], filtered)
        peaks = find_peaks(filtered, 0.005, distance=10, threshold=0.0)
        peak_positions = peaks[0] 
        peak_heights = peaks[1]["peak_heights"]
        ax.scatter(peak_positions, peak_heights, c="r")#, s=10)
        # ax.plot(bins[:-1], curve4(bins[:-1], *popt4))
        # ax.plot(bins[:-1], curve2(bins[:-1], *popt2))
        ax.set_title(key)
        ax.set_xticks(range(1, 255, 35))
        # print(f"{popt4 = }")
    fig.tight_layout()

    fig, ax=plt.subplots()
    ax.plot(bins[:-1], curve4(bins[:-1], *p04))
    ax.plot(bins[:-1], curve2(bins[:-1], *p02))

### using specific intensity of RGB values for ice / no-ice

In [14]:
run_live_tester = 0
def live_tester(start):
    # open video at beginning 
    i = start
    cap.set(cv2.CAP_PROP_POS_FRAMES, i) # search up the frame
    ret, frame = cap.read()

    fighistory, axshistory=plt.subplots(2, 1)
    low = []
    high = []

    while cap.isOpened():
        ret, frame = cap.read()
        cv2.imshow("..", frame)

        counts = extract_im_and_hist(cap, i, mask, False, searchup=False)
        ROI_B = None
        ROI_G = np.where(100<bins)[0][0], np.where(160<bins)[0][0]
        ROI_R = np.where(110<bins)[0][0], np.where(160<bins)[0][0]

        # count_B_ROI = counts["counts_blue"][ROI_B[0]:ROI_B[1]]
        count_G_ROI = counts["counts_green"][ROI_G[0]: ROI_G[1]]
        count_R_ROI = counts["counts_red"][ROI_R[0]: ROI_R[1]]
        mean_G = np.mean(count_G_ROI)
        mean_R = np.mean(count_R_ROI)
        low.append(mean_G)
        high.append(mean_R)

        axshistory[0].plot(low, c="g")
        axshistory[1].plot(high, c="r")
        fighistory.savefig(figure_dir / "history.png")

        print(f"{mean_G = :.4}, {mean_R = :.4}", end="\r")

        fig, axs = plt.subplots(3, 1, sharex=True, sharey=True)
        # axs[0].plot(bins[:-1], count_B_ROI)#counts["counts_blue"])
        axs[1].plot(bins[ROI_G[0]: ROI_G[1]], count_G_ROI)#counts["counts_green"])
        axs[2].plot(bins[ROI_R[0]: ROI_R[1]], count_R_ROI)#counts["counts_red"])
        for ax in axs: ax.set_ylim(0, 0.02)
        fig.savefig(figure_dir / "histogram.png")
        plt.close("all")


        key = cv2.waitKey(1)
        if key == ord("q"):
            break

        if key == ord("i"):
            axshistory[0].axvline(len(low)-1)
            axshistory[1].axvline(len(low)-1)

        elif key == ord(" "):
            k = cv2.waitKey(0)
            if k == ord(" "):
                continue
            elif k == ord("q"):
                break
        increase = 100
        for _ in range(increase - 1):
            # grab but dont decde
            cap.grab()    
        i += increase
    cv2.destroyAllWindows()
    
if run_live_tester:
    live_tester(0)

### run through video and split ice from no-ice

Using the regions of interest, we should be able to split the original video into two videos 

In [16]:
split_video_into_ice_noice = 1
if split_video_into_ice_noice:

    outputvideos_dir = Path("..") / "outputvideos"
    output_icing_path = outputvideos_dir / ("icing_" + video_filename)
    output_noicing_path = outputvideos_dir / ("noicing_" + video_filename)
    if not outputvideos_dir.is_dir(): outputvideos_dir.mkdir(parents=True, exist_ok=True)

    width, height = frame_shape[1], frame_shape[0]
    # from
    # https://learnopencv.com/reading-and-writing-videos-using-opencv/
    output_icing = cv2.VideoWriter(output_icing_path,
                                cv2.VideoWriter_fourcc(*'XVID'), 20, (width, height))
    output_noicing = cv2.VideoWriter(output_noicing_path,
                                    cv2.VideoWriter_fourcc(*'XVID'), 20, (width, height))





def split_into_ice_noice(increase=100, num_frames = 1000):
    # open video at beginning 
    i = 0
    cap.set(cv2.CAP_PROP_POS_FRAMES, i) # search up the frame
    ret, frame = cap.read()

    green_value = []
    red_value = []
    ROI_B = None
    ROI_G = np.where(100<bins)[0][0], np.where(160<bins)[0][0]
    ROI_R = np.where(110<bins)[0][0], np.where(160<bins)[0][0]
    green_threshold = 0.006
    red_threshold = 0.005

    fig, axs = plt.subplots(2, 1)
    num_dp = 1
    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break
        # cv2.imshow("..", frame)

        counts = extract_im_and_hist(cap, i, mask, False, searchup=False)
        if not counts:
            break
        # count_B_ROI = counts["counts_blue"][ROI_B[0]:ROI_B[1]]
        count_G_ROI = counts["counts_green"][ROI_G[0]: ROI_G[1]]
        count_R_ROI = counts["counts_red"][ROI_R[0]: ROI_R[1]]
        mean_G = np.mean(count_G_ROI)
        mean_R = np.mean(count_R_ROI)
        green_value.append(mean_G)
        red_value.append(mean_R)


        if mean_G > green_threshold and mean_R > red_threshold:
            print(f"{mean_G = :.4}, {mean_R = :.4}  icing       ", end="\r")
            output_icing.write(frame)
            axs[0].scatter(num_dp, mean_G, c="g", marker="x")
            axs[1].scatter(num_dp, mean_R, c="r", marker="x")

        else:
            print(f"{mean_G = :.4}, {mean_R = :.4}  no icing        ", end="\r")
            output_noicing.write(frame)
            axs[0].scatter(num_dp, mean_G, c="g")
            axs[1].scatter(num_dp, mean_R, c="r")

        fig.savefig(figure_dir / "long_history.png")
        plt.close()

        if num_dp == num_frames and num_frames:
            break

        for _ in range(increase):
            cap.grab()
        num_dp += 1
        i += increase

    output_icing.release()
    output_noicing.release()
    cv2.destroyAllWindows()

if split_video_into_ice_noice:
    split_into_ice_noice(500, 0)




In [ ]:
show_noicing = 1
show_icing = 1


cap_noicing = cv2.VideoCapture(output_noicing_path)
cap_icing = cv2.VideoCapture(output_icing_path)

total_frames_noicing = int(cap_noicing.get(cv2.CAP_PROP_FRAME_COUNT))
total_frames_icing = int(cap_icing.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"{total_frames_noicing = }")
print(f"{total_frames_icing = }")

cap_noicing.set(cv2.CAP_PROP_POS_FRAMES, 0)


def show_noicing_video():
    while cap_noicing.isOpened():
        ret, frame = cap_noicing.read()

        if not ret:
            break

        cv2.imshow("No Icing", frame)

        key = cv2.waitKey(0)
        if key == ord("q"):
            break

def show_icing_video():
    cap_icing.set(cv2.CAP_PROP_POS_FRAMES, 0)
    while cap_icing.isOpened():
        ret, frame = cap_icing.read()

        if not ret:
            break

        cv2.imshow("Icing", frame)

        key = cv2.waitKey(0)
        if key == ord("q"):
            break


if show_noicing:
    show_noicing_video()
if show_icing:
    show_icing_video()

cv2.destroyAllWindows()

total_frames_noicing = 127
total_frames_icing = 95
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-icing
no-i